# 46. 원판 검출을 ONNX로 옮겨도 같은 궤적을 얻을까?

원판 베이스라인 `best.pt`를 FP32 ONNX로 내보냅니다. 검증 6명의 영상은 0번부터 재생하고, 프레임마다 전처리한 텐서 하나를 두 런타임에 전달합니다. 이 영상들은 `imgsz=640`에서 세로 640 × 가로 384가 됩니다. 모든 프레임에서 실제 입력 모양을 검사합니다.

Mac의 프로젝트 커널에서 위에서 아래로 실행하세요. 이 실험의 추가 패키지는 프로젝트 루트에서 `uv pip install --python .venv/bin/python onnx==1.23.2 onnxruntime==1.32.0`으로 설치할 수 있습니다. 검증한 Ultralytics 버전은 8.4.202입니다.

## 1. 대상과 판정 기준 확인

박스 네 좌표의 최대 차이는 원본 영상에서 0.10 px 이하, 신뢰도의 최대 차이는 0.0001 이하로 정합니다. 동일한 후처리에서 후보 수와 최종 선택 출처도 일치해야 합니다. 44번에서 확인한 원본 라벨 56장, b02 35번 시간 연결, c04 11→27번 상승량을 다시 봅니다.

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import sys

start = Path(globals().get("__vsc_ipynb_file__", Path.cwd())).resolve()
root = next(p for p in (start, *start.parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display
from wlcv.config import VIDEOS
from wlcv.data.frames import read_rgb
from wlcv.evaluation.onnx_plate import BOX, WEIGHTS, evaluate, load_review, run_comparison

reference = root / "outputs/plate_saturation_review/20261125-191408-181503"
review = load_review(reference)
print(WEIGHTS, "\n", reference)
display(review.groupby("person_id").frame_index.agg(["min", "max", "count"]))

## 2. 그래프를 내보내고 원본 영상 재생

CPU, FP32, 배치 1, 연산 스레드 4개로 맞춥니다. 외부 NMS를 사용했던 원본 실행과 같은 검출 헤드입니다. 두 런타임에 같은 NMS·원본 좌표 복원·시간 연결을 적용합니다. 45번 크기 보정은 사용하지 않습니다. 영상마다 10번씩 예열 후 어느 쪽을 먼저 실행할지 번갈아 배치합니다. 지연에는 모델 호출과 CPU 원시 출력 반환만 포함합니다.

In [ ]:
folder = root / "outputs/onnx_plate_parity" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
folder.mkdir(parents=True)
detail, timing, tracks = run_comparison(folder, review)
print(folder)

## 3. 검출·궤적·지연 확인

전체 영상에서 NMS 후보의 좌표와 신뢰도, 최종 선택 출처가 달랐는지부터 확인합니다. 직접·연결·정탐 수는 수동 박스가 있는 56장의 값입니다. 좌표 참조 비교는 44번 원본을 사용합니다. 모델 원시 출력 최대 차이의 좌표 단위는 입력 픽셀입니다. 표의 p50·p95는 전체 영상에서 측정한 프레임당 지연 시간(ms)입니다.

In [ ]:
summary, scored, checks = evaluate(detail, timing, tracks, review)
display(summary.round(6))
print("b02 35번 · 원본 픽셀 좌표와 선택 점수")
case35 = scored[scored.video_id.eq("b02_sn_h_02") & scored.frame_index.eq(35)]
display(case35.set_index("runtime")[BOX + ["confidence", "source", "iou"]].round(9))
print(json.dumps(checks, indent=2))
summary.to_csv(folder / "summary.csv")
scored.to_csv(folder / "reviewed_56.csv", index=False)
(folder / "parity.json").write_text(json.dumps(checks, indent=2))
display(tracks.groupby(["runtime", "source"]).size().unstack(fill_value=0))
print("지연 시간 · 영상별 중앙값 (ms)")
display(timing.groupby(["video_id", "runtime"]).ms.median().unstack().round(2))

## 4. b02 35번과 지연 분포 확인

공통 수동 박스는 흰 점선, 선택한 박스는 파란 실선입니다. 라벨이 있는 장면과 전체 영상 측정의 분모를 구분하세요. 지연 누적분포에서 세로축 0.95에 이르는 지점이 p95입니다.

In [ ]:
video, frame = "b02_sn_h_02", 35
rgb = read_rgb(VIDEOS / f"{video}.mp4", frame)
case = scored[scored.video_id.eq(video) & scored.frame_index.eq(frame)].set_index("runtime")
fig, axes = plt.subplots(1, 2, figsize=(8, 4), layout="constrained", sharex=True, sharey=True)
for ax, name in zip(axes, ["pytorch", "onnx"], strict=True):
    row = case.loc[name]
    ax.imshow(rgb)
    for prefix, color, style in [("gt_", "white", "--"), ("", "#19d4ec", "-")]:
        x, y, x2, y2 = [row[prefix + col] for col in BOX]
        ax.add_patch(Rectangle((x,y), x2-x, y2-y, fill=False, ec=color, ls=style, lw=2))
    ax.set(xlim=(row.gt_x1-65,row.gt_x2+65), ylim=(row.gt_y2+65,row.gt_y1-65),
           title=f"{name} | {row.source} | {row.confidence:.6f}")
    ax.axis("off")
fig.suptitle("b02 frame 35 | white = manual, blue = selected")
fig.savefig(folder / "b02_f35.png", dpi=140)
plt.show()

fig, ax = plt.subplots(figsize=(7, 3.2), layout="constrained")
for name, group in timing.groupby("runtime"):
    ax.ecdf(group.ms, label=name)
ax.set(xlabel="CPU inference latency (ms/frame)", ylabel="Cumulative fraction", xscale="log")
ax.axhline(.95, color="grey", lw=.8, ls="--")
ax.legend()
fig.savefig(folder / "latency.png", dpi=140)
plt.show()

## 5. 내보낸 그래프 다시 보기

로컬 Netron에서 실행 폴더의 `plate.onnx`를 열어 입력 `images`와 출력 `output0`를 볼 수 있습니다. `settings.json`에는 실제 런타임·그래프 형태·가중치 해시를 저장합니다. `frames.csv`는 후보별 차이의 프레임 단위 최댓값, `tracks.csv`는 시간 연결 결과, `latency.csv`는 프레임별 원시 지연 측정입니다. 이 비교는 저장한 체크포인트·세로 영상·Mac CPU 실행 환경에 해당합니다.